### IBM QAOA Parameter Setting Analysis ###
This notebook produces tables and generates plots to analyze real IBM Qunatum hardware data while running QAOA on ensembles of the MaxCut problem. Various training strategies are utilized to determine the optimal parameters (angles of QAOA) before running them on hardware. The idea is to create a resource-cost estimation for these different parameter setting strategies and advise best practices.

In [14]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from src.Processing import QAOAHardware, QAOATraining
from src.Processing import set_data_path
from src.Processing import load_problem_instance
from src.Appox_Ratio_Calc import maxcut_approximation_ratio
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np  
from pathlib import Path

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Data Analysis for IBM Quantum Hardware Data ####

In [15]:
# Select graph to explore
graph_type = "heavy_hex"

# Set data path
data_dir = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data"
# Set instance path
inst_path = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/instances"

# Set hardware data path to graph type
hardware_data_path = set_data_path(data_dir, True, False, graph_type)
print(hardware_data_path)

# Set training data path to graph type
training_data_path = set_data_path(data_dir, False, True, graph_type)
print(training_data_path)

# # Load instance path
# instance_path = load_problem_instance("heavy_hex")
# print(instance_path)

# Select hardware instance parameters to explore:
p_list = [5, 10] # eg. QAOA depths for heavy_hex
num_nodes = 144 # eg. Problem size
instance_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] # eg. instance index (last index only)

# Load heavy_hex instances for the above parameter combination
instance_paths_hardware = []

for p in p_list:
    for instance in instance_list:
        instance_paths_hardware.extend(QAOAHardware.locate_hardware_instance(hardware_data_path, 
        graph_type, str(instance), str(num_nodes), str(p))) # extend ensures flat list
print(instance_paths_hardware)
print(len(instance_paths_hardware)) 

/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex
/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex
[PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/000N144HH73_5_d4am1h7nmdfs73ad2p90.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/001N144HH73_5_d4mkjm90i6jc73dgt05g.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/002N144HH73_5_d4mkjrt74pkc738940t0.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/003N144HH73_5_d4mkk0p0i6jc73dgt0fg.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/004N144HH73_5_d4mkk610i6jc73dgt0kg.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/005N144HH73_5_d4mkkbav0j9c73e6e1dg.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Set

In [16]:
all_instances_data_hardware = []

for instance_path in instance_paths_hardware:
    all_instances_data_hardware.extend(QAOAHardware.load_hardware_instance(instance_path))

print(all_instances_data_hardware)

# Expnad each object into data columns
all_data_dicts_hardware = []

for object in all_instances_data_hardware:
    data_dict_hardware = {

        "instance_name" : object.instance_name,
        "QPU_time" : object.QPU_time,
        "num_shots" : object.num_shots,
        "problem_class" : object.problem_class,
        "training_method" : object.training_method,
        "expected_energy" : object.expected_energy,
        "result_file" : object.result_file
    }
    all_data_dicts_hardware.append(data_dict_hardware)

[<src.Processing.QAOAHardware object at 0x7d283ed01fc0>, <src.Processing.QAOAHardware object at 0x7d283ed02530>, <src.Processing.QAOAHardware object at 0x7d283ed02290>, <src.Processing.QAOAHardware object at 0x7d283ed02650>, <src.Processing.QAOAHardware object at 0x7d283ed02bf0>, <src.Processing.QAOAHardware object at 0x7d283ed02770>, <src.Processing.QAOAHardware object at 0x7d283ed02590>, <src.Processing.QAOAHardware object at 0x7d283ed011e0>, <src.Processing.QAOAHardware object at 0x7d283ed006a0>, <src.Processing.QAOAHardware object at 0x7d283ed00df0>, <src.Processing.QAOAHardware object at 0x7d283ed00fd0>, <src.Processing.QAOAHardware object at 0x7d283ed012d0>, <src.Processing.QAOAHardware object at 0x7d283ed01b10>, <src.Processing.QAOAHardware object at 0x7d283ed02380>, <src.Processing.QAOAHardware object at 0x7d283ed02410>, <src.Processing.QAOAHardware object at 0x7d283ed01150>, <src.Processing.QAOAHardware object at 0x7d283ed01c00>, <src.Processing.QAOAHardware object at 0x7d283e

In [17]:
# Creata a DataFrame to document the extracted hardware data  
df_hardware = pd.DataFrame(all_data_dicts_hardware)
print(df_hardware.head())

  instance_name QPU_time num_shots problem_class training_method  \
0   000N144HH73     None      None        maxcut        F_MPS_10   
1   000N144HH73     None      None        maxcut         F_PP_10   
2   000N144HH73     None      None        maxcut        I_MPS_10   
3   000N144HH73     None      None        maxcut         I_PP_10   
4   000N144HH73     None      None        maxcut    TQA_PP_opt_5   

   expected_energy                           result_file  
0        27.752374  000N144HH73_MC_F_MPS_optBD24_10.json  
1        50.809508    000N144HH73_MC_F_PP_optMW6_10.json  
2        24.198928  000N144HH73_MC_I_MPS_optBD24_10.json  
3        50.991603    000N144HH73_MC_I_PP_optMW6_10.json  
4        47.329537   000N144HH73_MC_TQA_PP_optMW6_5.json  
